# Звіт по сегментації (Excel)

Збирає `output/segmentation_report.xlsx` з результатів `pipeline.ipynb`.
Запускати після pipeline (і score). Нічого не перенавчає — лише читає `output/`.

Аркуші: **Дані** · **Методологія** · **Сегменти**. Таблиця всіх клієнтів — окремим файлом (254 тис. рядків).

In [ ]:
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from IPython.display import display

import utils
importlib.reload(utils)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

## 0. Налаштування

In [ ]:
DATA_PATH = Path(r'C:\repo\data-science\Nazar Hnatyshyn\RETAIL\(DS-1052) Vehicle market analysis\df_for_clustering.xlsx')
SHEET_NAME = 0
OUTPUT_DIR = Path("output")
REPORT_PATH = OUTPUT_DIR / "segmentation_report.xlsx"
CLIENTS_PATH = OUTPUT_DIR / "segmentation_clients.xlsx"

EXAMPLE_USER_IDS = [5142061, 6319870]   # клієнти для прикладу AS IS → TO BE (якщо їх немає — оберуться автоматично)
WRITE_CLIENTS_FILE = True               # окремий Excel з усіма клієнтами

## 1. Дані та сегменти

In [ ]:
config = utils.load_json(OUTPUT_DIR / "segmentation_config.json")
tree_rules = utils.load_json(OUTPUT_DIR / "tree_rules.json")

raw = utils.load_raw(DATA_PATH, SHEET_NAME)
vehicles = utils.prepare_vehicle_data(raw)
customers = utils.aggregate_customers(vehicles, age_median=config["age_median"])
segments = utils.assign_final_segments(customers, tree_rules, config["ev_rule"], config["business_threshold"])

# Контроль: сегменти збігаються з результатом pipeline
reference = pd.read_csv(OUTPUT_DIR / "customer_segments.csv.gz", usecols=["user_id", "FINAL_SEGMENT"])
check = segments[["user_id", "FINAL_SEGMENT"]].merge(reference, on="user_id", suffixes=("", "_REF"))
assert len(check) == len(segments) and check["FINAL_SEGMENT"].eq(check["FINAL_SEGMENT_REF"]).all()
print(f"ТЗ: {len(vehicles):,} | клієнтів: {len(segments):,} | сегменти збігаються з pipeline")

## 2. Аркуш «Дані»

In [ ]:
n_cars = segments["PASSENGER_VEHICLE_COUNT"]
with_car = n_cars.gt(0)

scale = pd.DataFrame({
    "Показник": ["Кількість ТЗ", "з них легкових авто", "Кількість клієнтів",
                 "Клієнти з хоча б одним легковим авто", "Клієнти без легкового авто"],
    "Кількість": [len(vehicles), int(vehicles["IS_PASSENGER"].sum()), len(segments),
                  int(with_car.sum()), int((~with_car).sum())],
})

vehicles_per_client = pd.cut(segments["TOTAL_VEHICLE_COUNT"], [0, 1, 2, 3, 4, 5, np.inf],
                             labels=["1", "2", "3", "4", "5", "6+"]).value_counts().sort_index()
vehicles_per_client = pd.DataFrame({
    "ТЗ на клієнта": vehicles_per_client.index.astype(str),
    "Клієнтів": vehicles_per_client.values,
    "% клієнтів": vehicles_per_client.values / len(segments),
})

cars_per_client = pd.cut(n_cars[with_car], [0, 1, 2, np.inf], labels=["1", "2", "3+"]).value_counts().sort_index()
cars_per_client = pd.DataFrame({
    "Легкових авто на клієнта": cars_per_client.index.astype(str),
    "Клієнтів": cars_per_client.values,
    "% клієнтів з легковим авто": cars_per_client.values / with_car.sum(),
})

display(scale, vehicles_per_client, cars_per_client)

In [ ]:
# Приклад клієнтів: AS IS → TO BE
example_ids = [uid for uid in EXAMPLE_USER_IDS if uid in set(segments["user_id"])]
if len(example_ids) < 2:
    tree_clients = segments.loc[segments["SEGMENT_SOURCE"].eq("TREE")]
    example_ids = [
        tree_clients.loc[tree_clients["PASSENGER_VEHICLE_COUNT"].eq(1), "user_id"].iloc[0],
        tree_clients.loc[tree_clients["PASSENGER_VEHICLE_COUNT"].eq(2), "user_id"].iloc[0],
    ]
example_ids = [int(x) for x in example_ids]
print("Клієнти для прикладу:", example_ids)

ex_raw = raw.loc[raw["user_id"].isin(example_ids)]
ex_veh = vehicles.loc[vehicles["user_id"].isin(example_ids)]
ex_cli = segments.loc[segments["user_id"].isin(example_ids)]

as_is = pd.DataFrame({
    "user_id": ex_raw["user_id"], "Вид ТЗ": ex_raw["kind"], "Марка": ex_raw["BRAND_STD"],
    "Модель": ex_raw["MODEL_STD"], "Рік": ex_raw["production_year"], "Пальне": ex_raw["fuel_type"],
    "Тип кузова": ex_raw["body_type"], "Вартість, USD": ex_raw["MARKET_VALUE_USD"],
})
yes_no = {1: "так", 0: "ні"}
fuel_names = {"ELECTRIC": "EV", "HYBRID": "Hybrid", "PETROL": "Бензин", "DIESEL": "Дизель", "GAS": "Газ", "UNKNOWN": "Невідомо"}
class_names = {"PASSENGER": "Легкове", "COMMERCIAL": "Вантажне / спецтехніка", "BUS": "Автобус",
               "MOTO": "Мото", "TRAILER": "Причіп", "OTHER": "Інше"}
to_be_vehicle = pd.DataFrame({
    "user_id": ex_veh["user_id"], "Марка": ex_veh["BRAND_STD"], "Модель": ex_veh["MODEL_STD"],
    "Клас ТЗ": ex_veh["VEHICLE_CLASS"].map(class_names), "Вік, років": ex_veh["VEHICLE_AGE_CLEAN"],
    "Premium": ex_veh["IS_PREMIUM_BRAND"].map(yes_no), "SUV": ex_veh["IS_SUV"].map(yes_no),
    "Клас пального": ex_veh["FUEL_CLASS"].map(fuel_names), "Вартість, USD": ex_veh["MARKET_VALUE_USD"],
})
to_be_client = pd.DataFrame({
    "user_id": ex_cli["user_id"], "Усього ТЗ": ex_cli["TOTAL_VEHICLE_COUNT"],
    "Легкових авто": ex_cli["PASSENGER_VEHICLE_COUNT"], "Сумарна вартість, USD": ex_cli["TOTAL_VEHICLE_VALUE_USD"],
    "Середній вік, років": ex_cli["AVG_VEHICLE_AGE"], "% Premium": ex_cli["PREMIUM_SHARE"],
    "% SUV": ex_cli["SUV_SHARE"], "% EV": ex_cli["EV_SHARE"], "% Hybrid": ex_cli["HYBRID_SHARE"],
    "Сегмент": ex_cli["FINAL_SEGMENT"].map(lambda s: utils.SEGMENT_DISPLAY.get(s, s)),
})
display(as_is, to_be_vehicle, to_be_client)

## 3. Аркуш «Методологія»

In [ ]:
steps = utils.hierarchy_business_table(tree_rules, config["ev_rule"], config["business_threshold"])
# Сегменти-правила без жодного клієнта (напр. Other) не показуємо
present = set(segments["FINAL_SEGMENT"])
steps = steps.loc[steps["SEGMENT"].isin(present)].reset_index(drop=True)
display(steps[["Крок", "Кого стосується", "Умова", "Сегмент"]])

brands = pd.DataFrame({
    "Категорія": ["Luxury-бренди", "Premium-бренди"],
    "Марки": [", ".join(sorted(utils.LUXURY_BRANDS)), ", ".join(sorted(utils.PREMIUM_BRANDS))],
    "Кількість марок": [len(utils.LUXURY_BRANDS), len(utils.PREMIUM_BRANDS)],
})

definitions = pd.DataFrame([
    ("Легкове авто", "ТЗ з видом «легковий» (фургони з помилковим типом кузова «автовоз» теж вважаються легковими)."),
    ("Premium-авто", "Авто Premium- або Luxury-бренду (список марок — у таблиці нижче)."),
    ("SUV / кросовер", "Модель зі списку SUV-моделей (Tiguan, RAV4, X5, Q7, Sportage тощо) або тип кузова «кросовер / позашляховик»."),
    ("EV", "Тип пального — лише «електро»."),
    ("Hybrid", "Поєднання кількох видів пального: «бензин або газ», «електро або бензин» тощо."),
    ("Газ", "Тип пального — лише «газ»."),
    ("Легкий причіп", "Причіп повною масою до 3,5 т, у т. ч. причепи для легкових авто."),
    ("Важкий причіп", "Напівпричепи та причепи повною масою понад 3,5 т."),
    ("Причіп для човна", "Причіп з типом кузова «для перевезення човнів»."),
    ("Бізнес-транспорт", "Вантажні авто, спецтехніка, автобуси та важкі причепи."),
    ("Вартість", "Ринкова вартість легкових авто, USD."),
    (f"Вік авто", f"{utils.REFERENCE_YEAR} мінус рік випуску (не більше {utils.MAX_VEHICLE_AGE} років)."),
], columns=["Термін", "Як визначається"])
display(brands, definitions)

## 4. Аркуш «Сегменти»

In [ ]:
stats = utils.segment_statistics(segments, vehicles)
for name, table in stats.items():
    print(name)
    display(table)
assert abs(stats["main"]["Клієнтів"].sum() - len(segments)) == 0

## 5. Запис Excel

In [ ]:
wb = Workbook()
fmt_int = "#,##0"
fmt_usd = "#,##0"
fmt_pct = "0.0%"

# ---------- Аркуш 1: Дані
ws = wb.active
ws.title = "Дані"
utils.set_widths(ws, [2, 30, 18, 18, 18, 16, 16, 20, 18, 16, 22])
utils.write_title(ws, 1, 2, "СТАТИСТИКА ТА ТРАНСФОРМАЦІЯ ДАНИХ ДЛЯ СЕГМЕНТАЦІЇ", 10)
utils.write_table(ws, 3, 2, scale, "Таблиця 1. Масштаб даних", {"Кількість": fmt_int}, left_cols=("Показник",))
utils.write_table(ws, 3, 5, vehicles_per_client, "Таблиця 2. Кількість ТЗ на клієнта",
                  {"Клієнтів": fmt_int, "% клієнтів": fmt_pct})
utils.write_table(ws, 3, 9, cars_per_client, "Таблиця 3. Легкових авто на клієнта",
                  {"Клієнтів": fmt_int, "% клієнтів з легковим авто": fmt_pct})
row = 13
row = utils.write_table(ws, row, 2, as_is, "Таблиця 4. AS IS: сирі дані про ТЗ клієнта",
                        {"Вартість, USD": fmt_usd}) + 2
row = utils.write_table(ws, row, 2, to_be_vehicle, "Таблиця 5. TO BE: ознаки на рівні авто",
                        {"Вартість, USD": fmt_usd}) + 2
row = utils.write_table(ws, row, 2, to_be_client, "Таблиця 6. TO BE: ознаки на рівні клієнта (основа сегментації)",
                        {"Сумарна вартість, USD": fmt_usd, "Середній вік, років": "0.0", "% Premium": fmt_pct,
                         "% SUV": fmt_pct, "% EV": fmt_pct, "% Hybrid": fmt_pct}) + 2
n_with, n_without = int(with_car.sum()), int((~with_car).sum())
utils.write_note(ws, row, 2, [
    f"{len(segments):,} унікальних клієнтів".replace(",", " "),
    f"{n_with:,} клієнтів з хоча б одним легковим авто сегментуються через кластеризацію та правила".replace(",", " "),
    f"{n_without:,} клієнтів без легкового авто отримують сегмент за окремими правилами".replace(",", " "),
], 10)

# ---------- Аркуш 2: Методологія
ws = wb.create_sheet("Методологія")
utils.set_widths(ws, [2, 8, 30, 80, 32])
utils.write_title(ws, 1, 2, "МЕТОДОЛОГІЯ ПРИСВОЄННЯ СЕГМЕНТА", 4)
ws.merge_cells("B2:E2")
ws["B2"] = "Правила перевіряються зверху вниз: клієнт отримує сегмент першого правила, яке до нього підходить."
ws["B2"].alignment = utils._styles()["left"]
fills = [utils.GROUP_COLORS[utils.segment_group(s)] for s in steps["SEGMENT"]]
row = utils.write_table(ws, 4, 2, steps[["Крок", "Кого стосується", "Умова", "Сегмент"]],
                        "Покрокові правила", row_fills=fills, left_cols=("Умова", "Кого стосується", "Сегмент"), header_height=24) + 2
row = utils.write_table(ws, row, 3, definitions, "Визначення ознак",
                        left_cols=("Термін", "Як визначається"), header_height=24) + 2
utils.write_table(ws, row, 3, brands, "Списки марок", left_cols=("Категорія", "Марки"), height=48, header_height=24)

# ---------- Аркуш 3: Сегменти
ws = wb.create_sheet("Сегменти")
utils.set_widths(ws, [2, 18, 32, 62] + [14] * 14)
utils.write_title(ws, 1, 2, "СТАТИСТИКА ПО СЕГМЕНТАХ", 10)
order = list(stats["main"].index)
fills = [utils.GROUP_COLORS[utils.segment_group(s)] for s in order]
row = utils.write_table(ws, 3, 2, stats["main"], "Таблиця 1. Основні показники",
                        {"Клієнтів": fmt_int, "% бази": fmt_pct, "Кількість ТЗ": fmt_int, "Кількість легкових": fmt_int,
                         "Медіана вартості легкових, USD": fmt_usd, "Медіана вартості всіх ТЗ, USD": fmt_usd,
                         "Медіана віку легкових, років": "0.0"},
                        row_fills=fills, left_cols=("Група", "Сегмент", "Опис"), header_height=60)
main_header_row, main_last_row = 4, row - 1
row += 2

pct_formats = lambda df: {c: fmt_pct for c in df.columns if c not in ("Сегмент",) and not c.startswith(("Легкових", "Усього"))}
row = utils.write_table(ws, row, 4, stats["flags"], "Таблиця 2. % клієнтів сегмента з ознакою (кожна колонка окремо)",
                        pct_formats(stats["flags"]), row_fills=fills, left_cols=("Сегмент",)) + 2
cars_fmt = pct_formats(stats["cars"]) | {"Легкових авто": fmt_int}
row = utils.write_table(ws, row, 4, stats["cars"],
                        "Таблиця 3. Структура легкових авто сегмента: тип (100%) і пальне (100%)",
                        cars_fmt, row_fills=fills, left_cols=("Сегмент",)) + 2
fleet_fmt = pct_formats(stats["fleet"]) | {"Усього ТЗ": fmt_int}
utils.write_table(ws, row, 4, stats["fleet"], "Таблиця 4. Структура всіх ТЗ сегмента (100%)",
                  fleet_fmt, row_fills=fills, left_cols=("Сегмент",))

chart = BarChart()
chart.type = "bar"
chart.title = "Клієнтів у сегменті"
chart.legend = None
chart.height, chart.width = 9, 16
chart.add_data(Reference(ws, min_col=5, min_row=main_header_row, max_row=main_last_row), titles_from_data=True)
chart.set_categories(Reference(ws, min_col=3, min_row=main_header_row + 1, max_row=main_last_row))
chart.y_axis.scaling.orientation = "minMax"
chart.x_axis.scaling.orientation = "maxMin"
ws.add_chart(chart, f"M{main_header_row}")

for sheet in wb.worksheets:
    sheet.sheet_view.showGridLines = False
wb.save(REPORT_PATH)
print("Збережено:", REPORT_PATH.resolve())

## 6. Таблиця всіх клієнтів (окремий файл)

In [ ]:
if WRITE_CLIENTS_FILE:
    client_cols = ["user_id", "FINAL_SEGMENT", "TOTAL_VEHICLE_COUNT", "PASSENGER_VEHICLE_COUNT",
                   "TOTAL_VEHICLE_VALUE_USD", "AVG_VEHICLE_AGE", "PREMIUM_SHARE", "SUV_SHARE", "EV_SHARE",
                   "HYBRID_SHARE", "HAS_PREMIUM", "HAS_SUV", "HAS_EV", "HAS_HYBRID", "HAS_GAS", "HAS_LUXURY_VEHICLE",
                   "HAS_MODERN_PREMIUM", "HAS_TRAILER", "HAS_LIGHT_TRAILER", "HAS_HEAVY_TRAILER", "HAS_BOAT_TRAILER",
                   "HAS_CAMPER_TRAILER", "HAS_COMMERCIAL", "HAS_BUS", "HAS_MOTO", "BUSINESS_VEHICLE_COUNT", "TRAILER_COUNT"]
    clients = segments[client_cols].copy()
    clients.insert(2, "SEGMENT_GROUP", clients["FINAL_SEGMENT"].map(utils.segment_group))
    clients.insert(2, "SEGMENT_NAME", clients["FINAL_SEGMENT"].map(lambda s: utils.SEGMENT_DISPLAY.get(s, s)))
    clients["AVG_VEHICLE_AGE"] = clients["AVG_VEHICLE_AGE"].where(clients["PASSENGER_VEHICLE_COUNT"].gt(0))

    cw = Workbook(write_only=True)
    sheet = cw.create_sheet("Клієнти")
    sheet.freeze_panes = "B2"
    sheet.auto_filter.ref = f"A1:{chr(64 + len(clients.columns)) if len(clients.columns) <= 26 else 'AD'}{len(clients) + 1}"
    sheet.append(list(clients.columns))
    for values in clients.itertuples(index=False):
        sheet.append([None if isinstance(v, float) and np.isnan(v) else (v.item() if isinstance(v, np.generic) else v)
                      for v in values])
    cw.save(CLIENTS_PATH)
    print(f"Збережено: {CLIENTS_PATH.resolve()} ({len(clients):,} клієнтів)")